In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [ ]:
from scripts.preprocessing.pipeline import process

from scripts.preprocessing.feature_engeneering import (
    add_binary_service_or_item,
    add_global_category_and_city,
    add_date_features,
    aggregate_to_monthly,
    add_sparse_zero_rows_fast,
    add_lags_and_rolling,
    add_item_history_features,
    add_shop_aggregates,
    add_global_price_features,
    encode_categoricals_transform,
    prepare_test,
)

In [3]:
train, (sample_submission, test), dqc_report = process("../data", verbose=False)

Обнаружено 4023 записей с дублирующимися названиями после чистки!
Примеры дубликатов:
    item_id                      item_name
12       12  МИХЕЙ И ДЖУМАНДЖИ Сука любовь
30       30        007 КООРДИНАТЫ СКАЙФОЛЛ
31       31        007 КООРДИНАТЫ СКАЙФОЛЛ
32       32                             11
33       33                             11
35       35                  10 ЛЕТ СПУСТЯ
36       36                  10 ЛЕТ СПУСТЯ
37       37                  10 ЛЕТ СПУСТЯ
71       71            11 ДРУЗЕЙ ОУШЕНА WB
72       72            11 ДРУЗЕЙ ОУШЕНА WB

Найдено 1657 уникальных названий, которые повторяются:
                       item_name  count
0        007 КООРДИНАТЫ СКАЙФОЛЛ      2
1                  10 ЛЕТ СПУСТЯ      3
2                             11      2
3            11 ДРУЗЕЙ ОУШЕНА WB      2
4            12 ДРУЗЕЙ ОУШЕНА WB      2
5                 12 ЛЕТ РАБСТВА      2
6                      127 ЧАСОВ      3
7                   12ДВЕНАДЦАТЬ      2
8            13 ДРУЗЕЙ ОУ

In [4]:
with_service_or_item = add_binary_service_or_item(train)
with_service_or_item.head()

Successfuly passed service or item part


,date,date_block_num,shop_id,item_id,item_price,item_cnt_day,item_name,item_category_id,item_category_name,shop_name,service_or_item
0,2013-01-01,0,2,991,99.0,1.0,3D Action Puzzle Динозавры Тиранозавр,67,Подарки - Развитие,Адыгея ТЦ Мега,0
1,2013-01-01,0,2,1472,2599.0,1.0,Assassins Creed 3 Xbox 360 русская версия,23,Игры - XBOX 360,Адыгея ТЦ Мега,0
2,2013-01-01,0,2,1905,249.0,1.0,Bestseller. Grand Theft Auto San Andreas PC Jewel,30,Игры PC - Стандартные издания,Адыгея ТЦ Мега,0
3,2013-01-01,0,2,2920,599.0,2.0,Disney. LEGO Пираты Карибского моряPSP русская...,21,Игры - PSP,Адыгея ТЦ Мега,0
4,2013-01-01,0,2,3320,1999.0,1.0,FIFA 13PS3 русская версия,19,Игры - PS3,Адыгея ТЦ Мега,0


In [5]:
with_global_cat_and_city = add_global_category_and_city(with_service_or_item)
with_global_cat_and_city.head()

Successfuly passed global_category and shop_city part


,date,date_block_num,shop_id,item_id,item_price,item_cnt_day,item_name,item_category_id,item_category_name,shop_name,service_or_item,global_category,shop_city
0,2013-01-01,0,2,991,99.0,1.0,3D Action Puzzle Динозавры Тиранозавр,67,Подарки - Развитие,Адыгея ТЦ Мега,0,Подарки,Адыгея
1,2013-01-01,0,2,1472,2599.0,1.0,Assassins Creed 3 Xbox 360 русская версия,23,Игры - XBOX 360,Адыгея ТЦ Мега,0,Игры,Адыгея
2,2013-01-01,0,2,1905,249.0,1.0,Bestseller. Grand Theft Auto San Andreas PC Jewel,30,Игры PC - Стандартные издания,Адыгея ТЦ Мега,0,Игры PC,Адыгея
3,2013-01-01,0,2,2920,599.0,2.0,Disney. LEGO Пираты Карибского моряPSP русская...,21,Игры - PSP,Адыгея ТЦ Мега,0,Игры,Адыгея
4,2013-01-01,0,2,3320,1999.0,1.0,FIFA 13PS3 русская версия,19,Игры - PS3,Адыгея ТЦ Мега,0,Игры,Адыгея


In [6]:
with_date_features = add_date_features(with_global_cat_and_city)
with_date_features.head()

Successfuly passed data_features part


,date,date_block_num,shop_id,item_id,item_price,item_cnt_day,item_name,item_category_id,item_category_name,shop_name,service_or_item,global_category,shop_city,month_num,is_december,month_sin,month_cos
0,2013-01-01,0,2,991,99.0,1.0,3D Action Puzzle Динозавры Тиранозавр,67,Подарки - Развитие,Адыгея ТЦ Мега,0,Подарки,Адыгея,1,0,0.5,0.866025
1,2013-01-01,0,2,1472,2599.0,1.0,Assassins Creed 3 Xbox 360 русская версия,23,Игры - XBOX 360,Адыгея ТЦ Мега,0,Игры,Адыгея,1,0,0.5,0.866025
2,2013-01-01,0,2,1905,249.0,1.0,Bestseller. Grand Theft Auto San Andreas PC Jewel,30,Игры PC - Стандартные издания,Адыгея ТЦ Мега,0,Игры PC,Адыгея,1,0,0.5,0.866025
3,2013-01-01,0,2,2920,599.0,2.0,Disney. LEGO Пираты Карибского моряPSP русская...,21,Игры - PSP,Адыгея ТЦ Мега,0,Игры,Адыгея,1,0,0.5,0.866025
4,2013-01-01,0,2,3320,1999.0,1.0,FIFA 13PS3 русская версия,19,Игры - PS3,Адыгея ТЦ Мега,0,Игры,Адыгея,1,0,0.5,0.866025


In [7]:
aggregated_to_monthly = aggregate_to_monthly(with_date_features)
aggregated_to_monthly.head()

Successfuly passed month aggregation part


,date_block_num,shop_id,item_id,month_num,item_price_mean,item_cnt_month,global_category,shop_city,month_sin,month_cos,is_december,service_or_item
0,0,0,32,1,221.0,6.0,Кино,Якутск,0.5,0.866025,0,0
1,0,0,33,1,347.0,3.0,Кино,Якутск,0.5,0.866025,0,0
2,0,0,35,1,247.0,1.0,Кино,Якутск,0.5,0.866025,0,0
3,0,0,43,1,221.0,1.0,Кино,Якутск,0.5,0.866025,0,0
4,0,0,51,1,128.5,2.0,Музыка,Якутск,0.5,0.866025,0,0


In [16]:
sparsed = add_sparse_zero_rows_fast(aggregated_to_monthly)
sparsed.head()

Original rows: 1609124
New rows: 3195230


,date_block_num,shop_id,item_id,month_num,item_price_mean,item_cnt_month,global_category,shop_city,month_sin,month_cos,is_december,service_or_item
0,1,0,30,2.0,265.0,31.0,Кино,Якутск,0.866025,0.500000,0.0,0.0
1,1,0,31,2.0,434.0,11.0,Кино,Якутск,0.866025,0.500000,0.0,0.0
2,0,0,32,1.0,221.0,6.0,Кино,Якутск,0.500000,0.866025,0.0,0.0
3,1,0,32,2.0,221.0,10.0,Кино,Якутск,0.866025,0.500000,0.0,0.0
4,0,0,33,1.0,347.0,3.0,Кино,Якутск,0.500000,0.866025,0.0,0.0


In [17]:
lags_and_rolling = add_lags_and_rolling(sparsed)
lags_and_rolling.head()

Successfuly passed lags and rolling part


,date_block_num,shop_id,item_id,month_num,item_price_mean,item_cnt_month,global_category,shop_city,month_sin,month_cos,is_december,service_or_item,lag_1,lag_2,lag_3,lag_12,rolling_mean_3,had_sales_lag1
0,1,0,30,2.0,265.0,31.0,Кино,Якутск,0.866025,0.500000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,1,0,31,2.0,434.0,11.0,Кино,Якутск,0.866025,0.500000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,0,0,32,1.0,221.0,6.0,Кино,Якутск,0.500000,0.866025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,1,0,32,2.0,221.0,10.0,Кино,Якутск,0.866025,0.500000,0.0,0.0,6.0,0.0,0.0,0.0,6.0,1
4,0,0,33,1.0,347.0,3.0,Кино,Якутск,0.500000,0.866025,0.0,0.0,0.0,0.0,0.0,0.0,6.0,0


In [18]:
item_history = add_item_history_features(lags_and_rolling, 33)
item_history.head()

Successfuly passed item history features part


,date_block_num,shop_id,item_id,month_num,item_price_mean,item_cnt_month,global_category,shop_city,month_sin,month_cos,...,lag_2,lag_3,lag_12,rolling_mean_3,had_sales_lag1,first_month,last_month,month_with_sales,total_sales,avg_sales
0,1,0,30,2.0,265.0,31.0,Кино,Якутск,0.866025,0.500000,...,0.0,0.0,0.0,0.0,0,1.0,32.0,32.0,2089.0,2.097390
1,1,0,31,2.0,434.0,11.0,Кино,Якутск,0.866025,0.500000,...,0.0,0.0,0.0,0.0,0,1.0,32.0,32.0,1422.0,1.110070
2,0,0,32,1.0,221.0,6.0,Кино,Якутск,0.500000,0.866025,...,0.0,0.0,0.0,0.0,0,0.0,32.0,33.0,2072.0,1.574468
3,1,0,32,2.0,221.0,10.0,Кино,Якутск,0.866025,0.500000,...,0.0,0.0,0.0,6.0,1,0.0,32.0,33.0,2072.0,1.574468
4,0,0,33,1.0,347.0,3.0,Кино,Якутск,0.500000,0.866025,...,0.0,0.0,0.0,6.0,0,0.0,32.0,33.0,824.0,0.649842


In [19]:
shop_aggregations = add_shop_aggregates(item_history)
shop_aggregations.head()

Successfuly passed shop aggregations part


,date_block_num,shop_id,item_id,month_num,item_price_mean,item_cnt_month,global_category,shop_city,month_sin,month_cos,...,rolling_mean_3,had_sales_lag1,first_month,last_month,month_with_sales,total_sales,avg_sales,shop_total_sales_all_time,shop_num_active_items,shop_num_categories
0,1,0,30,2.0,265.0,31.0,Кино,Якутск,0.866025,0.500000,...,0.0,0,1.0,32.0,32.0,2089.0,2.097390,11705.0,3600,14
1,1,0,31,2.0,434.0,11.0,Кино,Якутск,0.866025,0.500000,...,0.0,0,1.0,32.0,32.0,1422.0,1.110070,11705.0,3600,14
2,0,0,32,1.0,221.0,6.0,Кино,Якутск,0.500000,0.866025,...,0.0,0,0.0,32.0,33.0,2072.0,1.574468,11705.0,3600,14
3,1,0,32,2.0,221.0,10.0,Кино,Якутск,0.866025,0.500000,...,6.0,1,0.0,32.0,33.0,2072.0,1.574468,11705.0,3600,14
4,0,0,33,1.0,347.0,3.0,Кино,Якутск,0.500000,0.866025,...,6.0,0,0.0,32.0,33.0,824.0,0.649842,11705.0,3600,14


In [20]:
gloabl_price_features = add_global_price_features(shop_aggregations)
gloabl_price_features.head()

Successfuly passed global price features part


,date_block_num,shop_id,item_id,month_num,item_cnt_month,global_category,shop_city,month_sin,month_cos,is_december,...,shop_num_active_items,shop_num_categories,item_price_global_mean,item_price_global_median,item_price_global_std,category_price_global_mean,category_price_global_median,price_ratio_to_category,is_expensive,is_cheap
0,1,0,30,2.0,31.0,Кино,Якутск,0.866025,0.500000,0.0,...,3600,14,252.022976,169.0,112.286632,366.306071,200.0,0.688012,0,0
1,1,0,31,2.0,11.0,Кино,Якутск,0.866025,0.500000,0.0,...,3600,14,520.605587,499.0,144.349474,366.306071,200.0,1.421231,0,0
2,0,0,32,1.0,6.0,Кино,Якутск,0.500000,0.866025,0.0,...,3600,14,205.210460,149.0,91.948928,366.306071,200.0,0.560216,0,1
3,1,0,32,2.0,10.0,Кино,Якутск,0.866025,0.500000,0.0,...,3600,14,205.210460,149.0,91.948928,366.306071,200.0,0.560216,0,1
4,0,0,33,1.0,3.0,Кино,Якутск,0.500000,0.866025,0.0,...,3600,14,246.945863,199.0,112.095390,366.306071,200.0,0.674152,0,0


# Пример с подвыборкой

==========================================


In [21]:
example_shop = 31
example_item = 13598

raw_example = train[
    (train["shop_id"] == example_shop) & (train["item_id"] == example_item)
]
print("Исходные дневные продажи:")
display(raw_example[["date", "date_block_num", "item_cnt_day", "item_price"]])

monthly_example = aggregated_to_monthly[
    (aggregated_to_monthly["shop_id"] == example_shop)
    & (aggregated_to_monthly["item_id"] == example_item)
]
print("После агрегации до месяца:")
display(monthly_example[["date_block_num", "item_cnt_month", "item_price_mean"]])

lags_example = lags_and_rolling[
    (lags_and_rolling["shop_id"] == example_shop)
    & (lags_and_rolling["item_id"] == example_item)
]
print("После добавления лагов (lag_1 — продажи за предыдущий месяц):")
display(
    lags_example[
        ["date_block_num", "item_cnt_month", "lag_1", "lag_2", "rolling_mean_3"]
    ]
)

history_example = item_history[
    (item_history["item_id"] == example_item)
].drop_duplicates("item_id")
print("История товара (одинакова для всех месяцев):")
display(history_example[["first_month", "last_month", "total_sales", "avg_sales"]])

price_example = gloabl_price_features[
    (gloabl_price_features["shop_id"] == example_shop)
    & (gloabl_price_features["item_id"] == example_item)
]
print("Финальные признаки (цена глобальная, is_expensive и т.д.):")
display(
    price_example[
        [
            "date_block_num",
            "item_price_global_mean",
            "is_expensive",
            "price_ratio_to_category",
        ]
    ]
)

Исходные дневные продажи:


,date,date_block_num,item_cnt_day,item_price
5433,2013-01-02,0,64.0,10.000000
12089,2013-01-03,0,8.0,10.000000
2457210,2015-02-20,25,11.0,20.000000
2461158,2015-02-21,25,12.0,20.000000
2465941,2015-02-22,25,7.0,20.000000
2470456,2015-02-23,25,15.0,20.000000
2473496,2015-02-24,25,5.0,20.000000
2475342,2015-02-25,25,6.0,20.000000
2477136,2015-02-26,25,6.0,20.000000
2479162,2015-02-27,25,10.0,20.000000


После агрегации до месяца:


,date_block_num,item_cnt_month,item_price_mean
37483,0,72.0,10.000000
1322773,25,80.0,20.000000
1363305,26,90.0,19.960234


После добавления лагов (lag_1 — продажи за предыдущий месяц):


,date_block_num,item_cnt_month,lag_1,lag_2,rolling_mean_3
1660068,0,72.0,0.0,0.0,0.000000
1660069,1,0.0,72.0,0.0,72.000000
1660070,2,0.0,0.0,72.0,36.000000
1660071,3,0.0,0.0,0.0,24.000000
1660072,4,0.0,0.0,0.0,0.000000
1660073,5,0.0,0.0,0.0,0.000000
1660074,6,0.0,0.0,0.0,0.000000
1660075,7,0.0,0.0,0.0,0.000000
1660076,8,0.0,0.0,0.0,0.000000
1660077,9,0.0,0.0,0.0,0.000000


История товара (одинакова для всех месяцев):


,first_month,last_month,total_sales,avg_sales
444640,0.0,26.0,347.0,8.897436


Финальные признаки (цена глобальная, is_expensive и т.д.):


,date_block_num,item_price_global_mean,is_expensive,price_ratio_to_category
1660068,0,10.357955,0,0.011955
1660069,1,10.357955,0,0.011955
1660070,2,10.357955,0,0.011955
1660071,3,10.357955,0,0.011955
1660072,4,10.357955,0,0.011955
1660073,5,10.357955,0,0.011955
1660074,6,10.357955,0,0.011955
1660075,7,10.357955,0,0.011955
1660076,8,10.357955,0,0.011955
1660077,9,10.357955,0,0.011955


==========================================

In [22]:
# encode_categoricals_fit(gloabl_price_features, encoder_dir="../encoders")

In [23]:
encoded_cat_features = encode_categoricals_transform(
    gloabl_price_features, encoder_dir="../encoders"
)
print(encoded_cat_features.columns)
encoded_cat_features.head()

Index(['date_block_num', 'shop_id', 'item_id', 'month_num', 'item_cnt_month',
       'global_category', 'shop_city', 'month_sin', 'month_cos', 'is_december',
       'service_or_item', 'lag_1', 'lag_2', 'lag_3', 'lag_12',
       'rolling_mean_3', 'had_sales_lag1', 'first_month', 'last_month',
       'month_with_sales', 'total_sales', 'avg_sales',
       'shop_total_sales_all_time', 'shop_num_active_items',
       'shop_num_categories', 'item_price_global_mean',
       'item_price_global_median', 'item_price_global_std',
       'category_price_global_mean', 'category_price_global_median',
       'price_ratio_to_category', 'is_expensive', 'is_cheap'],
      dtype='object')


,date_block_num,shop_id,item_id,month_num,item_cnt_month,global_category,shop_city,month_sin,month_cos,is_december,...,shop_num_active_items,shop_num_categories,item_price_global_mean,item_price_global_median,item_price_global_std,category_price_global_mean,category_price_global_median,price_ratio_to_category,is_expensive,is_cheap
0,1,0,30,2.0,31.0,11.0,29.0,0.866025,0.500000,0.0,...,3600,14,252.022976,169.0,112.286632,366.306071,200.0,0.688012,0,0
1,1,0,31,2.0,11.0,11.0,29.0,0.866025,0.500000,0.0,...,3600,14,520.605587,499.0,144.349474,366.306071,200.0,1.421231,0,0
2,0,0,32,1.0,6.0,11.0,29.0,0.500000,0.866025,0.0,...,3600,14,205.210460,149.0,91.948928,366.306071,200.0,0.560216,0,1
3,1,0,32,2.0,10.0,11.0,29.0,0.866025,0.500000,0.0,...,3600,14,205.210460,149.0,91.948928,366.306071,200.0,0.560216,0,1
4,0,0,33,1.0,3.0,11.0,29.0,0.500000,0.866025,0.0,...,3600,14,246.945863,199.0,112.095390,366.306071,200.0,0.674152,0,0


In [24]:
test_final = prepare_test(train, encoded_cat_features, test, encoder_dir="../encoders")

Successfuly passed data_features part
Successfuly passed service or item part
Successfuly passed global_category and shop_city part


In [25]:
print(test_final.columns)
test_final.head(5)

Index(['shop_id', 'item_id', 'date_block_num', 'month_num', 'is_december',
       'month_sin', 'month_cos', 'service_or_item', 'global_category',
       'shop_city', 'shop_total_sales_all_time', 'shop_num_active_items',
       'shop_num_categories', 'first_month', 'last_month', 'month_with_sales',
       'total_sales', 'avg_sales', 'lag_1', 'lag_2', 'lag_3', 'lag_12',
       'rolling_mean_3', 'had_sales_lag1', 'item_price_global_mean',
       'item_price_global_median', 'item_price_global_std',
       'category_price_global_mean', 'category_price_global_median',
       'price_ratio_to_category', 'is_expensive', 'is_cheap'],
      dtype='object')


,shop_id,item_id,date_block_num,month_num,is_december,month_sin,month_cos,service_or_item,global_category,shop_city,...,rolling_mean_3,had_sales_lag1,item_price_global_mean,item_price_global_median,item_price_global_std,category_price_global_mean,category_price_global_median,price_ratio_to_category,is_expensive,is_cheap
0,5,5037,34,11,0,-0.5,0.866025,0,5.0,3.0,...,0.000000,0.0,2002.887073,1999.000,601.348270,0.0,0.0,1.0,0,0
1,5,5320,34,11,0,-0.5,0.866025,0,-1.0,3.0,...,0.000000,0.0,0.000000,0.000,0.000000,0.0,0.0,1.0,0,0
2,5,5233,34,11,0,-0.5,0.866025,0,5.0,3.0,...,1.333333,1.0,826.937373,749.125,257.688972,0.0,0.0,1.0,0,0
3,5,5232,34,11,0,-0.5,0.866025,0,5.0,3.0,...,0.000000,0.0,781.604512,599.500,254.366822,0.0,0.0,1.0,0,0
4,5,5268,34,11,0,-0.5,0.866025,0,-1.0,3.0,...,0.000000,0.0,0.000000,0.000,0.000000,0.0,0.0,1.0,0,0


In [26]:
test_cols = set(test_final.columns)
train_cols = set(encoded_cat_features.columns)

print(f"Onlt in test: {test_cols - train_cols}")
print(f"Only in train: {train_cols - test_cols}")
print(f"Common cols: {len(train_cols & test_cols)}")

Onlt in test: set()
Only in train: {'item_cnt_month'}
Common cols: 32
